<a href="https://colab.research.google.com/github/omtx-ai/omtx/blob/main/examples/notebooks/lula_om_space_to_order.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>\n\n# LULA Om Accessible Space To Order\n\nScore Om Accessible Space with hosted LULA-2, select top rows, then create a Wallet Credits-funded Molecule Fulfillment order. The final order cell is guarded by `PLACE_ORDER = False` so you do not accidentally spend Wallet Credits.

## Install And Configure\n\nStore `OMTX_API_KEY` in Colab Secrets or set it as an environment variable.

In [ ]:
# !pip install "omtx[lula]>=2.0.20" polars

In [ ]:
from pathlib import Path\nfrom uuid import uuid4\n\nimport polars as pl\nfrom omtx import OmClient\n\nPROTEIN_SEQUENCE = "MEEPQSDPSV"\nTIER = 50\nN = 50_000\nTOP_K = 10_000\nSELECT = 100\nPLACE_ORDER = False

## Launch Hosted LULA-2 Scoring

In [ ]:
with OmClient() as client:\n    job = client.lula2.score(\n        protein_sequence=PROTEIN_SEQUENCE,\n        source="om",\n        tier=TIER,\n        n=N,\n        top_k=TOP_K,\n        idempotency_key=f"lula2-om-space-{uuid4()}",\n    )\njob

## Download Ranked Result Rows

In [ ]:
result_dir = Path("outputs/lula2-om-space")\nartifact_paths = []\n\nwith OmClient() as client:\n    for job_id in job["job_ids"]:\n        client.jobs.wait(job_id, poll_interval=5, timeout=3600)\n        artifact_paths.extend(\n            Path(path)\n            for path in client.jobs.download_all_artifacts(\n                job_id,\n                output_dir=result_dir / job_id,\n                overwrite=True,\n            )\n        )\n\nscore_tables = [\n    pl.read_parquet(path)\n    for path in artifact_paths\n    if path.suffix == ".parquet"\n]\n\ndef score_column(frame: pl.DataFrame) -> str:\n    for column in ("score", "lula2_score", "lula1_crossattention_v1_score"):\n        if column in frame.columns:\n            return column\n    for column in frame.columns:\n        if column.endswith("_score"):\n            return column\n    raise ValueError(f"No score column found. Columns: {frame.columns}")\n\nscore_rows = pl.concat(score_tables)\nscore_rows = score_rows.sort(score_column(score_rows), descending=True)\nselected_hits = score_rows.head(SELECT).to_dicts()\nscore_rows.head(10)

## Place The Order

In [ ]:
if not PLACE_ORDER:\n    print("Dry run. Set PLACE_ORDER = True to create a Wallet Credits-funded order.")\nelse:\n    with OmClient() as client:\n        addresses = client.molecules.shipping_addresses()\n        order = client.molecules.order(\n            items=selected_hits,\n            shipping_address_id=addresses["default_shipping_address_id"],\n            idempotency_key=f"molecule-order-{uuid4()}",\n        )\n    print(order["order_number"])